<a href="https://colab.research.google.com/github/athirasivadas94-glitch/AI-and-Gen-AI-Module-End-Assessment/blob/main/AI_and_Gen_AI_Module_End_Assessment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**AI and Gen AI Module End Assessment**

**AI-powered paraphrasing tool**

Develop an AI-powered paraphrasing tool as a Python application or module. The tool
should:
● Take a block of input text
● Use deep learning (preferably transformer models like T5, BERT, GPT, or similar
via Hugging Face Transformers)
● Generate a paraphrased version preserving meaning, improving clarity, and
ensuring originality.
● Include built-in checks for grammar, spelling, and fluency of output
Recommended technologies:
● Python, with Hugging Face Transformers and relevant NLP packages (spaCy,
NLTK, etc.)
● TensorFlow or PyTorch for any model fine-tuning
● Use only console or script-based input/output (no GUI or web interface)
● Optional: Integrate evaluation metrics like BLEU, ROUGE, or semantic similarity
scores
Expected Output:
● Well-commented Python scripts/notebooks
● Sample results demonstrating paraphrased outputs
● Evaluation report showing tool accuracy and originality

In [2]:
#Install Required Libraries
!pip install -q transformers torch sentencepiece
!pip install -q language-tool-python
!pip install -q nltk rouge-score sacrebleu sentence-transformers

In [3]:
# Import Libraries
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import language_tool_python
from sentence_transformers import SentenceTransformer, util
from rouge_score import rouge_scorer
from sacrebleu import sentence_bleu

In [14]:
#Load Models
## Paraphrasing Model
model_name = "humarin/chatgpt_paraphraser_on_T5_base"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

# Grammar Checker
grammar_tool = language_tool_python.LanguageTool('en-US')

# Semantic Similarity Model
similarity_model = SentenceTransformer(
    'sentence-transformers/all-MiniLM-L6-v2'
)

print("All models loaded successfully!")

config.json:   0%|          | 0.00/1.61k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.32k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/257 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


All models loaded successfully!


In [16]:
#Define Functions
def paraphrase_text(text, num_return_sequences=3):

    input_ids = tokenizer(
        text,
        return_tensors="pt",
        max_length=256,
        truncation=True
    )

    outputs = model.generate(
        input_ids=input_ids["input_ids"],
        attention_mask=input_ids["attention_mask"],
        max_length=256,
        num_beams=5,
        num_return_sequences=num_return_sequences,
        temperature=1.5,
        repetition_penalty=1.5,
        early_stopping=True
    )

    paraphrases = []

    for output in outputs:
        paraphrase = tokenizer.decode(
            output,
            skip_special_tokens=True
        )

        paraphrases.append(paraphrase)

    return paraphrases


In [17]:
# Grammar Correction Function
def grammar_check(text):

    matches = grammar_tool.check(text)

    corrected_text = language_tool_python.utils.correct(
        text,
        matches
    )

    return corrected_text

In [18]:
#Evaluation Function
def evaluate(original, paraphrased):

    emb1 = similarity_model.encode(
        original,
        convert_to_tensor=True
    )

    emb2 = similarity_model.encode(
        paraphrased,
        convert_to_tensor=True
    )

    semantic_similarity = util.cos_sim(
        emb1,
        emb2
    ).item()

    scorer = rouge_scorer.RougeScorer(
        ['rouge1', 'rougeL'],
        use_stemmer=True
    )

    rouge_scores = scorer.score(
        original,
        paraphrased
    )

    bleu_score = sentence_bleu(
        paraphrased,
        [original]
    ).score

    return semantic_similarity, rouge_scores, bleu_score

In [19]:
input_text = input("Enter text to paraphrase:\n")

Enter text to paraphrase:
Artificial Intelligence is transforming industries by automating tasks and improving decision-making processes.


In [20]:
#Generate Paraphrases
generated_paraphrases = paraphrase_text(input_text)

print("\nGenerated Paraphrases")
print("=" * 60)

for i, para in enumerate(generated_paraphrases, start=1):
    print(f"\nParaphrase {i}:")
    print(para)

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.



Generated Paraphrases

Paraphrase 1:
Artificial Intelligence is revolutionizing industries by automating tasks and improving decision-making processes.

Paraphrase 2:
Artificial Intelligence is revolutionizing industries by automating tasks and enhancing decision-making processes.

Paraphrase 3:
Artificial Intelligence is transforming industries by automating tasks and improving decision-making processes.


In [22]:
#Grammar Correction
corrected_paraphrases = []

print("\nGrammar Corrected Versions")
print("=" * 60)

for i, para in enumerate(generated_paraphrases, start=1):

    corrected = grammar_check(para)

    corrected_paraphrases.append(corrected)

    print(f"\nCorrected Paraphrase {i}:")
    print(corrected)


Grammar Corrected Versions

Corrected Paraphrase 1:
Artificial Intelligence is revolutionizing industries by automating tasks and improving decision-making processes.

Corrected Paraphrase 2:
Artificial Intelligence is revolutionizing industries by automating tasks and enhancing decision-making processes.

Corrected Paraphrase 3:
Artificial Intelligence is transforming industries by automating tasks and improving decision-making processes.


In [23]:
#Evaluate Best Paraphrase
best_paraphrase = corrected_paraphrases[0]

similarity, rouge, bleu = evaluate(
    input_text,
    best_paraphrase
)

print("\nEvaluation Results")
print("=" * 60)

print("Original Text:")
print(input_text)

print("\nSelected Paraphrase:")
print(best_paraphrase)

print("\nSemantic Similarity:",
      round(similarity, 4))

print("BLEU Score:",
      round(bleu, 2))

print("ROUGE-1:",
      round(rouge['rouge1'].fmeasure, 4))

print("ROUGE-L:",
      round(rouge['rougeL'].fmeasure, 4))


Evaluation Results
Original Text:
Artificial Intelligence is transforming industries by automating tasks and improving decision-making processes.

Selected Paraphrase:
Artificial Intelligence is revolutionizing industries by automating tasks and improving decision-making processes.

Semantic Similarity: 0.9741
BLEU Score: 76.12
ROUGE-1: 0.9231
ROUGE-L: 0.9231


In [24]:
#Test Samples for Report
samples = [

    "Machine learning helps computers learn from data.",

    "Online education has become increasingly popular in recent years.",

    "Artificial Intelligence is transforming industries by automating tasks and improving decision making.",

    "Climate change is one of the most significant challenges facing humanity today."
]

for sample in samples:

    print("\n" + "="*80)

    print("Original:")
    print(sample)

    paras = paraphrase_text(sample)

    best = grammar_check(paras[0])

    print("\nParaphrased:")
    print(best)

    similarity, rouge, bleu = evaluate(
        sample,
        best
    )

    print("\nSemantic Similarity:",
          round(similarity,4))

    print("BLEU:",
          round(bleu,2))

    print("ROUGE-L:",
          round(rouge['rougeL'].fmeasure,4))


Original:
Machine learning helps computers learn from data.

Paraphrased:
Machine learning assists computers in learning by acquiring knowledge from data.

Semantic Similarity: 0.8988
BLEU: 16.59
ROUGE-L: 0.6667

Original:
Online education has become increasingly popular in recent years.

Paraphrased:
The popularity of online education has risen in recent years.

Semantic Similarity: 0.9365
BLEU: 27.9
ROUGE-L: 0.6316

Original:
Artificial Intelligence is transforming industries by automating tasks and improving decision making.

Paraphrased:
Artificial Intelligence is revolutionizing the way industries operate by automating tasks and improving decision-making.

Semantic Similarity: 0.9625
BLEU: 34.79
ROUGE-L: 0.8148

Original:
Climate change is one of the most significant challenges facing humanity today.

Paraphrased:
Climate change is presently one of the most critical threats to humanity.

Semantic Similarity: 0.8627
BLEU: 29.78
ROUGE-L: 0.6667
